# Лабораторная работа №2
## Вариационные автокодировщики для моделирования временных рядов

## Цель работы

Изучить применение вариационных автокодировщиков к моделированию временных рядов на примере погодных данных и исследовать особенности непрерывного и дискретного латентного представления при реконструкции и генерации окон временного ряда.

## Задачи работы

1. Использовать набор погодных данных, применявшийся в лабораторной работе №1, и выделить временной ряд для выбранного города.
2. Выполнить предобработку данных и перейти от исходного временного ряда к набору скользящих окон фиксированной длины.
3. Реализовать и обучить модель Continuous VAE для реконструкции окон временного ряда.
4. Исследовать качество реконструкции, генерацию новых окон и интерполяцию в непрерывном латентном пространстве.
5. Реализовать и обучить модель VQ-VAE с дискретной кодовой книгой.
6. Проанализировать использование кодов в VQ-VAE и сопоставить наиболее частые коды с характерными формами окон.
7. Сравнить Continuous VAE и VQ-VAE по визуальному качеству реконструкции и по численным метрикам.
8. Показать проблему разрывов при независимой реконструкции окон и исследовать сглаживание методом overlap-add.
9. Сформулировать выводы о различии между авторегрессионным и генеративным подходами к моделированию временных рядов.

## Краткое теоретическое введение

### Автокодировщики

Автокодировщик состоит из двух частей: **энкодера** и **декодера**.  
Энкодер отображает входной объект $x$ в скрытое представление $z$:

$$
z = f_{\theta}(x),
$$

а декодер восстанавливает объект по скрытому коду:

$$
\hat{x} = g_{\phi}(z).
$$

При обучении обычного автокодировщика минимизируется ошибка реконструкции, например среднеквадратичная ошибка:

$$
\mathcal{L}_{AE}(x, \hat{x}) = \|x - \hat{x}\|_2^2.
$$

Однако обычный автокодировщик строит **детерминированное** скрытое представление.

### VAE

Вариационный автокодировщик (`VAE`) использует **вероятностную** постановку: вместо одного вектора $z$ модель задаёт апостериорное распределение

$$
q_{\phi}(z \mid x),
$$

которое обычно приближается многомерным нормальным распределением с диагональной ковариацией:

$$
q_{\phi}(z \mid x) = \mathcal{N}(z; \mu(x), \operatorname{diag}(\sigma^2(x))).
$$

Энкодер предсказывает параметры $\mu(x)$ и $\log \sigma^2(x)$, после чего выполняется выборка латентной переменной.  
Чтобы сделать выборку дифференцируемой, используется **трюк репараметризации**:

$$
z = \mu + \sigma \odot \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I).
$$

В декодере задаётся условное распределение данных при фиксированном скрытом векторе:

$$
p_{\theta}(x \mid z).
$$

Обучение VAE основано на максимизации вариационной нижней оценки правдоподобия \(ELBO\):

$$
\log p_{\theta}(x) \ge
\mathbb{E}_{q_{\phi}(z \mid x)}[\log p_{\theta}(x \mid z)]
- D_{KL}\bigl(q_{\phi}(z \mid x)\,\|\,p(z)\bigr),
$$

где $p(z)$ — априорное распределение, обычно

$$
p(z) = \mathcal{N}(0, I).
$$

На практике минимизируется функция потерь

$$
\mathcal{L}_{VAE}
=
\mathcal{L}_{rec}
+
\beta \, \mathcal{L}_{KL},
$$

где

$$
\mathcal{L}_{rec} = - \mathbb{E}_{q_{\phi}(z \mid x)}[\log p_{\theta}(x \mid z)]
$$

— ошибка реконструкции, а

$$
\mathcal{L}_{KL} =
D_{KL}\bigl(q_{\phi}(z \mid x)\,\|\,p(z)\bigr)
$$

— регуляризатор, заставляющий латентное пространство быть близким к априорному распределению.

Если $q_{\phi}(z \mid x)$ и $p(z)$ нормальны, то дивергенция Кульбака–Лейблера вычисляется аналитически:

$$
D_{KL}\bigl(\mathcal{N}(\mu, \sigma^2)\,\|\,\mathcal{N}(0,1)\bigr)
=
\frac{1}{2}\sum_{j=1}^{d}
\left(
\mu_j^2 + \sigma_j^2 - \log \sigma_j^2 - 1
\right).
$$

Для временных рядов VAE обычно применяется не к отдельным наблюдениям, а к **окнам фиксированной длины**.  
Если исходный ряд обозначить как

$$
x_1, x_2, \dots, x_T,
$$

то из него формируются окна длины $L$:

$$
X^{(i)} = \bigl(x_i, x_{i+1}, \dots, x_{i+L-1}\bigr), \qquad i = 1, 2, \dots, T-L+1.
$$

Таким образом, модель учится не предсказывать следующий элемент ряда, а **кодировать и восстанавливать целый фрагмент** временной последовательности.

В работе рассматриваются два варианта генеративной модели.

### Continuous VAE

В Continuous VAE латентное пространство является непрерывным. Это позволяет:
- выполнять плавную интерполяцию между окнами;
- генерировать новые окна выборкой $z \sim \mathcal{N}(0, I)$;
- анализировать структуру скрытого пространства.

Интерполяция между двумя латентными кодами $z_a$ и $z_b$ задаётся формулой

$$
z(\alpha) = (1-\alpha) z_a + \alpha z_b, \qquad \alpha \in [0,1].
$$

Затем по каждому $z(\alpha)$ декодируется новое окно:

$$
\hat{x}(\alpha) = g_{\theta}(z(\alpha)).
$$

Одной из типичных проблем Continuous VAE является **posterior collapse** — ситуация, когда латентные переменные почти перестают использоваться, а декодер восстанавливает усреднённый сигнал. В этом случае KL-слагаемое становится слишком малым, а реконструкция стремится к среднему профилю окна.

### Discrete VAE и VQ-VAE

В дискретном варианте (`VQ-VAE`) энкодер формирует непрерывный вектор $z_e(x)$, после чего он заменяется ближайшим кодом из кодовой книги

$$
\mathcal{E} = \{e_1, e_2, \dots, e_K\}.
$$

Выбор кода осуществляется по правилу ближайшего соседа:

$$
k^* = \arg\min_{k} \| z_e(x) - e_k \|_2^2,
$$

а квантизованное представление определяется как

$$
z_q(x) = e_{k^*}.
$$

Декодер восстанавливает окно по дискретному представлению:

$$
\hat{x} = g_{\theta}(z_q(x)).
$$

Функция потерь VQ-VAE включает три составляющие:

$$
\mathcal{L}_{VQ}
=
\mathcal{L}_{rec}
+
\| \operatorname{sg}[z_e(x)] - e \|_2^2
+
\beta \| z_e(x) - \operatorname{sg}[e] \|_2^2,
$$

где:
- $\mathcal{L}_{rec}$ — ошибка реконструкции;
- второе слагаемое — подстройка кодовой книги;
- третье слагаемое — `commitment loss`, заставляющий энкодер не уходить слишком далеко от выбранного кода;
- $\operatorname{sg}[\cdot]$ — операция остановки градиента.

Дискретное латентное пространство удобно тем, что позволяет выявлять **типовые режимы** поведения ряда. Частота использования кодов показывает, какие шаблоны чаще встречаются в данных.

### Проблема стыков окон

Если длинный временной ряд восстанавливается по отдельным окнам независимо, то при их объединении могут появляться разрывы на границах.  
Для уменьшения этого эффекта используют перекрывающиеся окна и усреднение значений на пересечениях. Если $\hat{x}^{(m)}_t$ — значение, восстановленное из $m$-го окна в точке $t$, а $c_t$ — число окон, покрывающих позицию $t$, то итоговая склейка определяется как

$$
\hat{x}_t = \frac{1}{c_t} \sum_{m: \, t \in W_m} \hat{x}^{(m)}_t.
$$

Такой подход называется **overlap-add** и позволяет уменьшить искусственные скачки между соседними окнами.

Таким образом, вариационные автокодировщики позволяют перейти от задачи локального прогноза к задаче **генеративного моделирования временного ряда**, где интерес представляют реконструкция фрагментов, свойства латентного пространства, генерация новых окон и выявление скрытых режимов сигнала.

## Подключение библиотек и определение параметров

Подключаем библиотеки для работы с данными, визуализации и построения нейросетевых моделей.

In [ ]:
import os
import random
import warnings
from typing import Dict, List, Optional, Sequence, Tuple

import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from IPython.display import display

print("Библиотеки успешно загружены")

Определяем параметры обучения и настройки моделей.

In [ ]:
CONFIG = {
    "seed": 42,
    "data": {
        "features": ["rain", "solar_radiation", "temp_max"],
        "target": "temp_max",
        "continuous_features": ["solar_radiation", "temp_max"]
    },
    "split": {
        "train_ratio": 0.70,
        "val_ratio": 0.15
    },
    "window": {
        "size": 32,
        "stride": 1
    },
    "train": {
        "batch_size": 128
    },
    "cvae": {
        "latent_dim": 8,
        "hidden_dims": (256, 128, 64),
        "lr": 3e-4,
        "epochs": 200,
        "beta_max": 0.02,
        "warmup_epochs": 20,
        "beta_values": [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
    },
    "vqvae": {
        "embedding_dim": 8,
        "num_embeddings": 32,
        "num_latent_tokens": 4,
        "commitment_beta": 0.25,
        "lr": 1e-3,
        "epochs": 200,
        "hidden_dims": (256, 128, 64)
    },
    "plots": {
        "max_examples": 5,
        "max_windows_preview": 6,
        "random_state": 42,
        "fragment_len": 220
    }
}

FEATURES = CONFIG["data"]["features"]
TARGET = CONFIG["data"]["target"]
WINDOW_SIZE = CONFIG["window"]["size"]
BATCH_SIZE = CONFIG["train"]["batch_size"]
CVAE_CFG = CONFIG["cvae"]
VQVAE_CFG = CONFIG["vqvae"]
PLOT_CFG = CONFIG["plots"]

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Используемое устройство:", device)


## Загрузка данных

В работе анализируется датасет [Pakistan Weather Data (2014-2023)](https://www.kaggle.com/datasets/abdulwadood11220/pakistan-weather-data-2014-2023), который содержит информацию о погоде в Пакистане в разных городах. Для каждых даты и города имеется информация о максимальной и минимальной температурах, факта наличия дождя и величины солнечной радиации.

In [ ]:
def load_weather_data(file_id: str, data_path: str) -> pd.DataFrame:
    folder = os.path.dirname(data_path)
    if folder:
        os.makedirs(folder, exist_ok=True)

    if not os.path.exists(data_path):
        print("Загружаем weather.csv ...")
        gdown.download(id=file_id, output=data_path, quiet=False)
    else:
        print("Файл уже существует:", data_path)

    return pd.read_csv(data_path)

df_full = load_weather_data(
    file_id="1SSXm1Yldoxb_je803ou1NKqVS-cHuDMp",
    data_path="/content/weather.csv"
)

print("Размер исходного датасета:", df_full.shape)
display(df_full.head())


## Предобработка данных

Оставляем только город Rawalpindi, сортируем наблюдения по дате, бинаризуем признак дождя и выбираем только некоторые признаки.

In [ ]:
df = df_full.loc[df_full["city"] == "Rawalpindi", ["date"] + FEATURES].copy()
df["date"] = pd.to_datetime(df["date"]).dt.date
df = df.sort_values("date").dropna().reset_index(drop=True)
df["rain"] = (df["rain"] > 0).astype(np.float32)
df = df.set_index("date")

print(f"Количество записей для города Rawalpindi: {len(df)}")
display(df.head())
display(df.describe().T)


## Визуализация исходного ряда

Перед обучением модели полезно посмотреть на динамику признаков во времени. Это помогает понять масштаб значений, сезонность и поведение признаков.

In [ ]:
def plot_dataset(data: pd.DataFrame) -> None:
    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

    axes[0].plot(data.index, data["temp_max"], color="crimson")
    axes[0].set_title("Максимальная температура temp_max")
    axes[0].set_ylabel("°C")

    axes[1].plot(data.index, data["solar_radiation"], color="darkorange")
    axes[1].set_title("Солнечная радиация")
    axes[1].set_ylabel("radiation")

    axes[2].plot(data.index, data["rain"], color="royalblue")
    axes[2].set_title("Бинарный признак дождя")
    axes[2].set_ylabel("0/1")
    axes[2].set_xlabel("Дата")

    plt.tight_layout()
    plt.show()

plot_dataset(df)


## Разбиение на train, validation и test

Разбиваем ряд по времени на обучающую, валидационную и тестовую части, масштабируем непрерывные признаки.

In [ ]:
CONT_IDX = [
    FEATURES.index(name) for name in CONFIG["data"]["continuous_features"]
]
TEMP_IDX = FEATURES.index(TARGET)
TEMP_POS_IN_SCALER = CONT_IDX.index(TEMP_IDX)

def split_scale_time_series(data: pd.DataFrame,
                            features: List[str],
                            cont_idx: List[int],
                            train_ratio: float,
                            val_ratio: float) -> Dict[str, np.ndarray]:
    x = data[features].to_numpy(dtype=np.float32)
    dates = np.array(data.index)

    train_end = int(train_ratio * len(x))
    val_end = int((train_ratio + val_ratio) * len(x))

    x_train = x[:train_end].copy()
    x_val = x[train_end:val_end].copy()
    x_test = x[val_end:].copy()

    train_dates = dates[:train_end]
    val_dates = dates[train_end:val_end]
    test_dates = dates[val_end:]

    scaler = StandardScaler()
    x_train[:, cont_idx] = scaler.fit_transform(x_train[:, cont_idx])
    x_val[:, cont_idx] = scaler.transform(x_val[:, cont_idx])
    x_test[:, cont_idx] = scaler.transform(x_test[:, cont_idx])

    return {
        "X_train": x_train,
        "X_val": x_val,
        "X_test": x_test,
        "train_dates": train_dates,
        "val_dates": val_dates,
        "test_dates": test_dates,
        "scaler": scaler
    }

Реализуем функцию обратного преобразования отмасштабированных данных.

In [ ]:
def inverse_scale_feature(x: np.ndarray,
                          scaler: StandardScaler,
                          feature_pos: int) -> np.ndarray:
    x = np.asarray(x)
    flat_x = x.reshape(-1, 1)
    full = np.zeros((flat_x.shape[0], scaler.n_features_in_),
                    dtype=np.float32)
    full[:, feature_pos] = flat_x[:, 0]
    restored = scaler.inverse_transform(full)
    return restored[:, feature_pos].reshape(x.shape)

Визуализируем разбиение данных.

In [ ]:
def plot_scaled_feature_split(x_train: np.ndarray,
                              x_val: np.ndarray,
                              x_test: np.ndarray,
                              feature_idx: int,
                              feature_name: str) -> None:
    train_part = x_train[:, feature_idx]
    val_part = x_val[:, feature_idx]
    test_part = x_test[:, feature_idx]

    train_end = len(train_part)
    val_end = train_end + len(val_part)
    total_len = val_end + len(test_part)

    plt.figure(figsize=(16, 5))
    plt.plot(np.arange(train_end),
             train_part,
             label="Train",
             color="seagreen")

    plt.plot(np.arange(train_end, val_end),
             val_part,
             label="Val",
             color="darkorange")

    plt.plot(np.arange(val_end, total_len),
             test_part,
             label="Test",
             color="crimson")

    plt.title(f"Масштабированный признак {feature_name}")
    plt.xlabel("Индекс")
    plt.ylabel("Scaled value")
    plt.legend()
    plt.show()

split_data = split_scale_time_series(
    data=df,
    features=FEATURES,
    cont_idx=CONT_IDX,
    train_ratio=CONFIG["split"]["train_ratio"],
    val_ratio=CONFIG["split"]["val_ratio"]
)

X_train = split_data["X_train"]
X_val = split_data["X_val"]
X_test = split_data["X_test"]
train_dates = split_data["train_dates"]
val_dates = split_data["val_dates"]
test_dates = split_data["test_dates"]
scaler = split_data["scaler"]

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

plot_scaled_feature_split(X_train, X_val, X_test, TEMP_IDX, TARGET)


## Подготовка скользящих окон

В лабораторной работе модели обрабатывают не отдельные значения временного ряда, а окна фиксированной длины. Сформируем окна для train, validation и test.

In [ ]:
def create_sliding_windows(series: np.ndarray,
                           window_size: int,
                           stride: int = 1) -> Dict[str, np.ndarray]:
    tail_shape = series.shape[1:]
    if len(series) < window_size:
        shape = (0, window_size) + tail_shape
        return {
            "windows": np.empty(shape, dtype=np.float32),
            "starts": np.empty(0, dtype=np.int64),
            "ends": np.empty(0, dtype=np.int64)
        }

    starts = np.arange(0, len(series) - window_size + 1, stride)
    ends = starts + window_size
    windows = np.lib.stride_tricks.sliding_window_view(
        series, window_shape=window_size, axis=0)[::stride]
    windows = np.moveaxis(windows, -1, 1)

    return {
        "windows": windows.astype(np.float32, copy=False),
        "starts": starts,
        "ends": ends
    }

In [ ]:
def build_window_sets(train_series: np.ndarray,
                      val_series: np.ndarray,
                      test_series: np.ndarray,
                      window_size: int) -> Dict[str, Dict[str, np.ndarray]]:
    return {
        "train": create_sliding_windows(train_series, window_size, stride=1),
        "val": create_sliding_windows(val_series, window_size, stride=1),
        "test": create_sliding_windows(test_series, window_size, stride=1),
        "test_nonoverlap": create_sliding_windows(test_series,
                                                  window_size,
                                                  stride=window_size)
    }

Приведём несколько примеров сдвинутых температурных окон.

In [ ]:
def plot_sample_windows(windows: np.ndarray,
                        starts: np.ndarray,
                        feature_idx: int,
                        feature_name: str,
                        scaler: Optional[StandardScaler] = None,
                        feature_pos_in_scaler: Optional[int] = None,
                        max_windows: int = 6,
                        random_state: Optional[int] = None) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.ravel()
    num_windows = min(max_windows, len(windows), len(axes))

    if num_windows == 0:
        return

    rng = np.random.default_rng(random_state)
    if len(windows) > num_windows:
        start_idx = rng.integers(0, len(windows) - num_windows + 1)
    else:
        start_idx = 0

    window_indices = np.arange(start_idx, start_idx + num_windows)

    for axis_idx, window_idx in enumerate(window_indices):
        window = windows[window_idx]
        feature_window = window[:, feature_idx]

        if scaler is not None and feature_pos_in_scaler is not None:
            feature_window = inverse_scale_feature(
                feature_window, scaler, feature_pos_in_scaler)

        axes[axis_idx].plot(feature_window, color="teal")
        axes[axis_idx].set_title(
            f"Окно #{window_idx}, start={starts[window_idx]}")
        axes[axis_idx].set_xlabel("Шаг внутри окна")
        axes[axis_idx].set_ylabel(feature_name)
        axes[axis_idx].grid(True, linestyle='--', alpha=0.4)

    for axis_idx in range(num_windows, len(axes)):
        axes[axis_idx].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
scaled_windows = build_window_sets(X_train, X_val, X_test, WINDOW_SIZE)
train_windows = scaled_windows["train"]["windows"]
train_starts = scaled_windows["train"]["starts"]
val_windows = scaled_windows["val"]["windows"]
val_starts = scaled_windows["val"]["starts"]
test_windows = scaled_windows["test"]["windows"]
test_starts = scaled_windows["test"]["starts"]
test_windows_nonoverlap = scaled_windows["test_nonoverlap"]["windows"]

print("train_windows:", train_windows.shape)
print("val_windows:  ", val_windows.shape)
print("test_windows: ", test_windows.shape)

plot_sample_windows(
    windows=train_windows,
    starts=train_starts,
    feature_idx=TEMP_IDX,
    feature_name=TARGET,
    scaler=scaler,
    feature_pos_in_scaler=TEMP_POS_IN_SCALER,
    max_windows=PLOT_CFG["max_windows_preview"],
    random_state=PLOT_CFG["random_state"]
)

## Подготовка DataLoader

Сформируем объекты `DataLoader` для подачи данных в модель по мини-батчам. Это стандартный шаг перед обучением нейросетей в `PyTorch`.

In [ ]:
def build_loaders(train_windows: np.ndarray,
                  val_windows: np.ndarray,
                  test_windows: np.ndarray,
                  batch_size: int = 128) -> Dict[str, DataLoader]:

    def to_tensor(arr: np.ndarray) -> torch.Tensor:
        arr = np.array(arr, dtype=np.float32, copy=True)
        return torch.from_numpy(arr)

    return {
        "train": DataLoader(
            TensorDataset(to_tensor(train_windows)),
            batch_size=batch_size,
            shuffle=True
        ),
        "val": DataLoader(
            TensorDataset(to_tensor(val_windows)),
            batch_size=batch_size,
            shuffle=False
        ),
        "test": DataLoader(
            TensorDataset(to_tensor(test_windows)),
            batch_size=batch_size,
            shuffle=False
        )
    }

In [ ]:
loaders = build_loaders(train_windows,
                        val_windows,
                        test_windows,
                        batch_size=CONFIG["train"]["batch_size"])

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]

print("DataLoader готовы.")

## Continuous VAE: архитектура модели

Определим условный вариационный автокодировщик, который восстанавливает температурное окно по латентному коду и остальным признакам окна.


In [ ]:
class ConditionalVAE(nn.Module):
    def __init__(self,
                 window_size: int,
                 num_features: int,
                 target_idx: int,
                 latent_dim: int = 8,
                 hidden_dims: Tuple[int, ...] = (256, 128, 64)) -> None:
        super().__init__()

        if num_features < 2:
            raise ValueError("num_features must be >= 2")

        if not 0 <= target_idx < num_features:
            raise ValueError("target_idx is out of range")

        if len(hidden_dims) == 0:
            raise ValueError("hidden_dims must not be empty")

        self.window_size = window_size
        self.num_features = num_features
        self.target_idx = target_idx
        self.cond_idx = tuple(
            idx for idx in range(num_features) if idx != target_idx)

        self.target_dim = window_size
        self.cond_dim = window_size * len(self.cond_idx)
        self.latent_dim = latent_dim
        self.hidden_dims = tuple(hidden_dims)

        encoder_input_dim = self.target_dim + self.cond_dim
        decoder_input_dim = self.latent_dim + self.cond_dim

        self.encoder = self._make_encoder(
            input_dim=encoder_input_dim,
            hidden_dims=self.hidden_dims,
        )
        self.fc_mu = nn.Linear(self.hidden_dims[-1], latent_dim)
        self.fc_logvar = nn.Linear(self.hidden_dims[-1], latent_dim)

        self.decoder = self._make_decoder(
            input_dim=decoder_input_dim,
            hidden_dims=self.hidden_dims,
            output_dim=self.target_dim,
        )

    @staticmethod
    def _make_encoder(input_dim: int,
                      hidden_dims: Tuple[int, ...]) -> nn.Sequential:
        layers: List[nn.Module] = []
        in_dim = input_dim

        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            in_dim = h

        return nn.Sequential(*layers)

    @staticmethod
    def _make_decoder(input_dim: int,
                      hidden_dims: Tuple[int, ...],
                      output_dim: int) -> nn.Sequential:
        layers: List[nn.Module] = []
        in_dim = input_dim

        for h in reversed(hidden_dims):
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            in_dim = h

        layers.append(nn.Linear(in_dim, output_dim))
        return nn.Sequential(*layers)

    @staticmethod
    def flatten_batch(x: torch.Tensor) -> torch.Tensor:
        return x.flatten(start_dim=1)

    def split_inputs(self, x: torch.Tensor) -> Tuple[torch.Tensor,
                                                     torch.Tensor]:
        target = x[..., self.target_idx]
        condition = x[..., self.cond_idx]
        return target, condition

    def encode(self,
               target: torch.Tensor,
               condition: torch.Tensor) -> Tuple[torch.Tensor,
                                                 torch.Tensor]:
        enc_input = torch.cat(
            [self.flatten_batch(target), self.flatten_batch(condition)],
            dim=1
        )
        hidden = self.encoder(enc_input)
        mu = self.fc_mu(hidden)
        logvar = self.fc_logvar(hidden)
        return mu, logvar

    @staticmethod
    def reparameterize(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor, condition: torch.Tensor) -> torch.Tensor:
        dec_input = torch.cat([z, self.flatten_batch(condition)], dim=1)
        recon = self.decoder(dec_input)
        return recon.view(-1, self.window_size)

    def forward(self,
                x: torch.Tensor,
                sample: bool = True
                ) -> Tuple[torch.Tensor, torch.Tensor,
                           torch.Tensor, torch.Tensor]:
        target, condition = self.split_inputs(x)
        mu, logvar = self.encode(target, condition)
        z = self.reparameterize(mu, logvar) if sample else mu
        recon = self.decode(z, condition)
        return recon, target, mu, logvar

    def reconstruct(self, x:
                    torch.Tensor,
                    sample: bool = False) -> torch.Tensor:
        recon, _, _, _ = self.forward(x, sample=sample)
        return recon

    def sample_prior(self, condition: torch.Tensor) -> torch.Tensor:
        batch_size = condition.size(0)
        z = torch.randn(batch_size, self.latent_dim, device=condition.device)
        return self.decode(z, condition)

Функция потерь `CVAE` формируется из ошибки реконструкции и KL-регуляризации.

In [ ]:
def cvae_loss_function(recon_target: torch.Tensor,
                       target: torch.Tensor,
                       mu: torch.Tensor,
                       logvar: torch.Tensor,
                       beta: float = 0.02) -> Tuple[torch.Tensor,
                                                    torch.Tensor,
                                                    torch.Tensor]:
    recon_loss = ((recon_target - target).pow(2).sum(dim=1)).mean()
    kl_loss = 0.5 * (mu.pow(2) + logvar.exp() - 1.0 - logvar).sum(dim=1).mean()
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss

## Continuous VAE: обучение модели

Определим фунцию обучения автокодировщика для одной эпохи.

In [ ]:
def run_cvae_epoch(model: nn.Module,
                   loader: DataLoader,
                   device: torch.device,
                   beta: float,
                   optimizer: Optional[torch.optim.Optimizer] = None
                   ) -> Dict[str, float]:
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_sum = 0.0
    recon_sum = 0.0
    kl_sum = 0.0
    num_samples = 0

    for (x_batch,) in loader:
        x_batch = x_batch.to(device)

        with torch.set_grad_enabled(is_train):
            recon_batch, target_batch, mu, logvar = (
                model(x_batch, sample=is_train))
            total_loss, recon_loss, kl_loss = cvae_loss_function(
                recon_batch, target_batch, mu, logvar, beta=beta)

            if is_train:
                optimizer.zero_grad()
                total_loss.backward()
                optimizer.step()

        batch_size = x_batch.size(0)
        total_sum += total_loss.item() * batch_size
        recon_sum += recon_loss.item() * batch_size
        kl_sum += kl_loss.item() * batch_size
        num_samples += batch_size

    return {
        "total": total_sum / num_samples,
        "recon": recon_sum / num_samples,
        "kl": kl_sum / num_samples
    }

Определим фунцию полного обучения вариационного автокодировщика с формированием истории обучения.

In [ ]:
def train_conditional_vae(model: nn.Module,
                          train_loader: DataLoader,
                          val_loader: DataLoader,
                          device: torch.device,
                          epochs: int,
                          lr: float,
                          beta_max: float,
                          warmup_epochs: int) -> Dict[str, List[float]]:
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {
        "beta": [],
        "train_total": [],
        "train_recon": [],
        "train_kl": [],
        "val_total": [],
        "val_recon": [],
        "val_kl": []
    }

    model.to(device)

    for epoch in range(1, epochs + 1):
        beta = beta_max * min(1.0, epoch / max(1, warmup_epochs))

        train_metrics = run_cvae_epoch(
            model, train_loader, device, beta, optimizer)
        val_metrics = run_cvae_epoch(
            model, val_loader, device, beta, optimizer=None)

        history["beta"].append(beta)
        history["train_total"].append(train_metrics["total"])
        history["train_recon"].append(train_metrics["recon"])
        history["train_kl"].append(train_metrics["kl"])
        history["val_total"].append(val_metrics["total"])
        history["val_recon"].append(val_metrics["recon"])
        history["val_kl"].append(val_metrics["kl"])

        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:03d}/{epochs} | "
                f"beta={beta:.4f} | "
                f"train_total={train_metrics['total']:.4f} | "
                f"val_total={val_metrics['total']:.4f} | "
                f"train_recon={train_metrics['recon']:.4f} | "
                f"train_kl={train_metrics['kl']:.4f}"
            )

    return history

Визуализирем процесс обучения.

In [ ]:
def plot_cvae_history(history: Dict[str, List[float]]) -> None:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].plot(history["train_total"], label="train")
    axes[0].plot(history["val_total"], label="val")
    axes[0].set_title("Total loss")
    axes[0].legend()

    axes[1].plot(history["train_recon"], label="train")
    axes[1].plot(history["val_recon"], label="val")
    axes[1].set_title("Reconstruction loss")
    axes[1].legend()

    axes[2].plot(history["train_kl"], label="train")
    axes[2].plot(history["val_kl"], label="val")
    axes[2].set_title("KL loss")
    axes[2].legend()

    axes[3].plot(history["beta"], color="black")
    axes[3].set_title("Beta schedule")

    plt.tight_layout()
    plt.show()

In [ ]:
cvae = ConditionalVAE(
    window_size=WINDOW_SIZE,
    num_features=len(FEATURES),
    target_idx=TEMP_IDX,
    latent_dim=CVAE_CFG["latent_dim"],
    hidden_dims=CVAE_CFG["hidden_dims"]
).to(device)

cvae_history = train_conditional_vae(
    model=cvae,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=CVAE_CFG["epochs"],
    lr=CVAE_CFG["lr"],
    beta_max=CVAE_CFG["beta_max"],
    warmup_epochs=CVAE_CFG["warmup_epochs"]
)

plot_cvae_history(cvae_history)

## Continuous VAE: оценка, генерация и интерполяция

Определим функции получения предсказания VAE и вычисления метрик.


In [ ]:
def predict_cvae(model: nn.Module,
                 loader: DataLoader,
                 device: torch.device,
                 sample: bool = False) -> np.ndarray:
    predictions = []
    model.eval()
    with torch.inference_mode():
        for (x_batch,) in loader:
            recon_batch = model.reconstruct(x_batch.to(device), sample=sample)
            predictions.append(recon_batch.cpu().numpy())
    return np.concatenate(predictions, axis=0)

def extract_target_windows(loader: DataLoader, target_idx: int) -> np.ndarray:
    targets = []
    for (x_batch,) in loader:
        targets.append(x_batch[..., target_idx].cpu().numpy())
    return np.concatenate(targets, axis=0)

def regression_metrics(y_true: np.ndarray,
                       y_pred: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

In [ ]:
def evaluate_cvae(model: nn.Module,
                  loader: DataLoader,
                  device: torch.device,
                  target_idx: int,
                  scaler: StandardScaler,
                  target_pos_in_scaler: int,
                  sample: bool = False) -> Dict[str, object]:
    pred_scaled = predict_cvae(model, loader, device, sample=sample)
    true_scaled = extract_target_windows(loader, target_idx)

    true = inverse_scale_feature(true_scaled, scaler, target_pos_in_scaler)
    pred = inverse_scale_feature(pred_scaled, scaler, target_pos_in_scaler)

    return {
        "true_scaled": true_scaled,
        "pred_scaled": pred_scaled,
        "true": true,
        "pred": pred,
        "metrics": regression_metrics(true, pred)
    }

In [ ]:
test_results = evaluate_cvae(
    model=cvae,
    loader=test_loader,
    device=device,
    target_idx=TEMP_IDX,
    scaler=scaler,
    target_pos_in_scaler=TEMP_POS_IN_SCALER,
    sample=False
)

display(pd.DataFrame([test_results["metrics"]]))

Определим функции реконструкции данных на тестовых окнах и визуализации ошибок.

In [ ]:
def plot_window_reconstructions(results: Dict[str, np.ndarray],
                                title: str,
                                starts: Optional[np.ndarray] = None,
                                max_examples: int = 5,
                                random_state: Optional[int] = None) -> None:
    true_windows = results["true"]
    pred_windows = results["pred"]

    num_examples = min(max_examples, len(true_windows))
    if num_examples == 0:
        return

    rng = np.random.default_rng(random_state)
    window_indices = np.sort(rng.choice(len(true_windows),
                                        size=num_examples,
                                        replace=False))

    fig, axes = plt.subplots(num_examples, 1, figsize=(14, 2.8 * num_examples))
    axes = np.atleast_1d(axes)

    for axis, window_idx in zip(axes, window_indices):
        axis.plot(true_windows[window_idx],
                  label="Метка",
                  color="black",
                  linewidth=2)

        axis.plot(pred_windows[window_idx],
                  label="Реконструкция",
                  color="crimson",
                  linestyle="--",
                  linewidth=2)

        if starts is None:
            axis.set_title(f"{title} | окно #{window_idx}")
        else:
            axis.set_title(
                f"{title} | окно #{window_idx}, start={starts[window_idx]}")

        axis.set_xlabel("Шаг внутри окна")
        axis.set_ylabel(TARGET)
        axis.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_scatter_and_errors(y_true: np.ndarray,
                            y_pred: np.ndarray,
                            title_prefix: str) -> None:
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    errors = y_true - y_pred

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    min_value = min(y_true.min(), y_pred.min())
    max_value = max(y_true.max(), y_pred.max())

    axes[0].scatter(y_true, y_pred, alpha=0.2, s=12)
    axes[0].plot([min_value, max_value],
                 [min_value, max_value],
                 "r--",
                 linewidth=2)

    axes[0].set_title(f"{title_prefix}: Метка vs реконструкция")
    axes[0].set_xlabel("Метка")
    axes[0].set_ylabel("Реконструкция")

    axes[1].hist(errors,
                 bins=40,
                 color="steelblue",
                 edgecolor="black",
                 alpha=0.8)

    axes[1].set_title(f"{title_prefix}: распределение ошибок")
    axes[1].set_xlabel("Ошибка")
    axes[1].set_ylabel("Частота")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_window_reconstructions(
    test_results,
    "Conditional VAE",
    starts=test_starts,
    max_examples=PLOT_CFG["max_examples"],
    random_state=PLOT_CFG["random_state"]
)

In [ ]:
plot_scatter_and_errors(test_results["true"],
                        test_results["pred"],
                        "Conditional VAE")

Определим функции генерации из априорного распределения и интерполяции между двумя окнами в латентном пространстве.

In [ ]:
def plot_cvae_prior_samples(model: nn.Module,
                            windows: np.ndarray,
                            device: torch.device,
                            scaler: StandardScaler,
                            target_pos_in_scaler: int,
                            max_windows: int = 6,
                            num_samples: int = 3,
                            random_state: Optional[int] = None) -> None:
    num_windows = min(max_windows, len(windows))
    if num_windows == 0:
        return

    rng = np.random.default_rng(random_state)
    window_indices = np.sort(rng.choice(len(windows),
                                        size=num_windows,
                                        replace=False))

    fig, axes = plt.subplots(2, 3, figsize=(24, 8))
    axes = axes.ravel()
    sample_colors = ["purple", "crimson", "teal"]

    model.eval()
    with torch.inference_mode():
        for axis_idx, window_idx in enumerate(window_indices):
            x = torch.from_numpy(
                np.asarray(
                    windows[window_idx:window_idx + 1],
                    dtype=np.float32)
                ).to(device)
            true_target, condition = model.split_inputs(x)
            true_window = inverse_scale_feature(
                true_target.cpu().numpy()[0], scaler, target_pos_in_scaler)

            axes[axis_idx].plot(true_window,
                                color="black",
                                linewidth=2.5,
                                label="Метка")

            for sample_idx in range(num_samples):
                sampled_target = model.sample_prior(condition)
                sampled_window = inverse_scale_feature(
                    sampled_target.cpu().numpy()[0],
                    scaler,
                    target_pos_in_scaler
                )
                axes[axis_idx].plot(
                    sampled_window,
                    color=sample_colors[sample_idx % len(sample_colors)],
                    linewidth=1.8,
                    alpha=0.9,
                    label=f"Сэмпл #{sample_idx + 1}"
                )

            axes[axis_idx].set_title(f"Окно #{window_idx}")
            axes[axis_idx].set_xlabel("Шаг внутри окна")
            axes[axis_idx].set_ylabel(TARGET)
            axes[axis_idx].legend()
            axes[axis_idx].grid(True, linestyle="--", alpha=0.4)

    for axis_idx in range(num_windows, len(axes)):
        axes[axis_idx].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
def plot_cvae_interpolation(model: nn.Module,
                            windows: np.ndarray,
                            device: torch.device,
                            scaler: StandardScaler,
                            target_pos_in_scaler: int,
                            idx_a: Optional[int] = None,
                            idx_b: Optional[int] = None,
                            num_steps: int = 9,
                            random_state: Optional[int] = None) -> None:
    num_windows = len(windows)
    if num_windows == 0:
        return

    if num_windows == 1:
        idx_a, idx_b = 0, 0
    else:
        rng = np.random.default_rng(random_state)
        if idx_a is None or idx_b is None:
            idx_a, idx_b = rng.choice(num_windows, size=2, replace=False)
        elif idx_a == idx_b:
            raise ValueError("idx_a and idx_b must be different")

    x_a = torch.from_numpy(
        np.asarray(windows[idx_a:idx_a + 1], dtype=np.float32)
    ).to(device)

    x_b = torch.from_numpy(
        np.asarray(windows[idx_b:idx_b + 1], dtype=np.float32)
    ).to(device)

    model.eval()
    with torch.inference_mode():
        target_a, cond_a = model.split_inputs(x_a)
        target_b, cond_b = model.split_inputs(x_b)
        mu_a, _ = model.encode(target_a, cond_a)
        mu_b, _ = model.encode(target_b, cond_b)
        alphas = torch.linspace(0.0, 1.0, num_steps, device=device)
        decoded = []

        for alpha in alphas:
            z = (1.0 - alpha) * mu_a + alpha * mu_b
            cond = (1.0 - alpha) * cond_a + alpha * cond_b
            recon = model.decode(z, cond).cpu().numpy()[0]
            decoded.append(recon)

    decoded = np.asarray(decoded)
    decoded = inverse_scale_feature(decoded, scaler, target_pos_in_scaler)
    orig_a = inverse_scale_feature(
        target_a.cpu().numpy()[0], scaler, target_pos_in_scaler)
    orig_b = inverse_scale_feature(
        target_b.cpu().numpy()[0], scaler, target_pos_in_scaler)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5),
                             gridspec_kw={"width_ratios": [1.4, 1.0]})
    colors = plt.cm.plasma(np.linspace(0.1, 0.9, num_steps))

    axes[0].plot(orig_a, color="black", linewidth=2.5,
                 label=f"A, idx={idx_a}")
    axes[0].plot(orig_b, color="dimgray", linewidth=2.5,
                 linestyle="--", label=f"B, idx={idx_b}")

    for step_idx, alpha in enumerate(alphas.cpu().numpy()):
        axes[0].plot(decoded[step_idx],
                     color=colors[step_idx],
                     linewidth=1.8,
                     alpha=0.95,
                     label=f"α={alpha:.2f}")

    axes[0].set_title("Интерполяция окон температуры")
    axes[0].set_xlabel("Шаг внутри окна")
    axes[0].set_ylabel(TARGET)
    axes[0].legend(ncol=2, fontsize=9)
    axes[0].grid(True, linestyle="--", alpha=0.4)

    image = axes[1].imshow(decoded,
                           aspect="auto",
                           origin="lower",
                           cmap="plasma")
    axes[1].set_title("Heatmap интерполяции")
    axes[1].set_xlabel("Шаг внутри окна")
    axes[1].set_ylabel("Шаг интерполяции")
    axes[1].set_yticks(np.arange(num_steps))
    axes[1].set_yticklabels([f"{alpha:.2f}" for alpha in alphas.cpu().numpy()])
    plt.colorbar(image, ax=axes[1], label=TARGET)
    axes[1].grid(False)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_cvae_prior_samples(
    cvae,
    test_windows,
    device,
    scaler,
    TEMP_POS_IN_SCALER,
    max_windows=PLOT_CFG["max_windows_preview"],
    num_samples=3,
    random_state=PLOT_CFG["random_state"]
)

In [ ]:
plot_cvae_interpolation(
    cvae,
    test_windows,
    device,
    scaler,
    TEMP_POS_IN_SCALER,
    num_steps=9
)


## Эксперимент с разными значениями $\beta$

В этой части проверяется, как коэффициент при KL-слагаемом влияет на реконструкцию и структуру латентного пространства. Такой эксперимент помогает увидеть компромисс между качеством восстановления и регуляризацией.


In [ ]:
def collect_cvae_outputs(model: nn.Module,
                         loader: DataLoader,
                         device: torch.device,
                         sample: bool = False) -> Dict[str, np.ndarray]:
    recon_list, target_list, mu_list, logvar_list = [], [], [], []
    model.eval()

    with torch.inference_mode():
        for (x_batch,) in loader:
            x_batch = x_batch.to(device)
            recon_batch, target_batch, mu, logvar = (
                model(x_batch, sample=sample)
            )
            recon_list.append(recon_batch.cpu().numpy())
            target_list.append(target_batch.cpu().numpy())
            mu_list.append(mu.cpu().numpy())
            logvar_list.append(logvar.cpu().numpy())

    return {
        "recon": np.concatenate(recon_list, axis=0),
        "target": np.concatenate(target_list, axis=0),
        "mu": np.concatenate(mu_list, axis=0),
        "logvar": np.concatenate(logvar_list, axis=0)
    }

In [ ]:
def get_cvae_val_stats(model: nn.Module,
                       loader: DataLoader,
                       device: torch.device,
                       scaler: StandardScaler,
                       target_pos_in_scaler: int,
                       sample: bool = False) -> Dict[str, float]:
    outputs = collect_cvae_outputs(
        model, loader, device, sample=sample)
    y_true = inverse_scale_feature(
        outputs["target"], scaler, target_pos_in_scaler).reshape(-1)
    y_pred = inverse_scale_feature(
        outputs["recon"], scaler, target_pos_in_scaler).reshape(-1)
    val_mse = mean_squared_error(y_true, y_pred)
    mean_kl = 0.5 * np.mean(
        np.sum(
            np.exp(outputs["logvar"]) \
                + outputs["mu"] ** 2 \
                - 1.0 \
                - outputs["logvar"],
            axis=1
        )
    )
    return {
        "val_MSE": val_mse,
        "mean_KL": mean_kl
    }

In [ ]:
def run_beta_experiment(beta_values: List[float],
                        train_loader: DataLoader,
                        val_loader: DataLoader,
                        device: torch.device,
                        window_size: int,
                        num_features: int,
                        target_idx: int,
                        latent_dim: int,
                        hidden_dims: Tuple[int, ...],
                        lr: float,
                        scaler: StandardScaler,
                        target_pos_in_scaler: int,
                        epochs: int,
                        random_seed: int = 42) -> List[Dict[str, float]]:
    beta_results = []

    for beta in beta_values:
        print(f"\n=== Обучение Conditional beta-VAE, beta={beta} ===")
        set_seed(random_seed)

        beta_model = ConditionalVAE(
            window_size=window_size,
            num_features=num_features,
            target_idx=target_idx,
            latent_dim=latent_dim,
            hidden_dims=hidden_dims
        ).to(device)

        beta_history = train_conditional_vae(
            model=beta_model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            epochs=epochs,
            lr=lr,
            beta_max=beta,
            warmup_epochs=1
        )

        stats = get_cvae_val_stats(
            model=beta_model,
            loader=val_loader,
            device=device,
            scaler=scaler,
            target_pos_in_scaler=target_pos_in_scaler,
            sample=False
        )

        beta_results.append({
            "beta": beta,
            "final_train_recon": beta_history["train_recon"][-1],
            "final_train_kl": beta_history["train_kl"][-1],
            "final_val_recon": beta_history["val_recon"][-1],
            "final_val_kl": beta_history["val_kl"][-1],
            "val_MSE_original_scale": stats["val_MSE"],
            "mean_KL_per_window": stats["mean_KL"]
        })

    return beta_results

In [ ]:
beta_results = run_beta_experiment(
    beta_values=CVAE_CFG["beta_values"],
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    window_size=WINDOW_SIZE,
    num_features=len(FEATURES),
    target_idx=TEMP_IDX,
    latent_dim=CVAE_CFG["latent_dim"],
    hidden_dims=CVAE_CFG["hidden_dims"],
    lr=CVAE_CFG["lr"],
    scaler=scaler,
    target_pos_in_scaler=TEMP_POS_IN_SCALER,
    epochs=CVAE_CFG["epochs"],
    random_seed=CONFIG["seed"]
)

beta_df = pd.DataFrame(beta_results)
display(beta_df)

In [ ]:
def plot_beta_experiment_results(beta_df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    axes = axes.ravel()

    metrics = [
        ("final_train_recon", "Final train reconstruction"),
        ("final_val_recon", "Final val reconstruction"),
        ("final_train_kl", "Final train KL"),
        ("final_val_kl", "Final val KL"),
        ("val_MSE_original_scale", "Validation MSE in original scale"),
        ("mean_KL_per_window", "Mean KL per window")
    ]

    for ax, (col, title) in zip(axes, metrics):
        ax.plot(beta_df["beta"], beta_df[col], marker="o")
        ax.set_title(title)
        ax.set_xlabel("beta")
        ax.set_ylabel(col)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_beta_experiment_results(beta_df)

## Conditional VQ-VAE: архитектура модели

Теперь реализуется дискретный вариант автокодировщика с кодовой книгой. Эта модель заменяет непрерывные латентные векторы ближайшими дискретными кодами и позволяет анализировать часто встречающиеся шаблоны окон.


In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self,
                 num_embeddings: int,
                 embedding_dim: int,
                 commitment_beta: float = 0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_beta = commitment_beta

        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / num_embeddings,
                                            1.0 / num_embeddings)

    def forward(self, z_e: torch.Tensor) -> Dict[str, torch.Tensor]:
        original_shape = z_e.shape
        flat_z = z_e.reshape(-1, self.embedding_dim)
        codebook = self.embedding.weight

        distances = (
            flat_z.pow(2).sum(dim=1, keepdim=True)
            + codebook.pow(2).sum(dim=1)
            - 2.0 * flat_z @ codebook.t()
        )

        encoding_indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(encoding_indices).reshape(original_shape)

        codebook_loss = F.mse_loss(z_q, z_e.detach())
        commitment_loss = self.commitment_beta * F.mse_loss(z_e, z_q.detach())
        vq_loss = codebook_loss + commitment_loss
        z_q_st = z_e + (z_q - z_e).detach()

        encodings_onehot = F.one_hot(encoding_indices,
                                     num_classes=self.num_embeddings).float()
        avg_probs = encodings_onehot.mean(dim=0)
        perplexity = torch.exp(
            -(avg_probs * torch.log(avg_probs + 1e-10)).sum()
        )
        encoding_indices = encoding_indices.reshape(*original_shape[:-1])

        return {
            "z_q_st": z_q_st,
            "vq_loss": vq_loss,
            "perplexity": perplexity,
            "indices": encoding_indices,
            "codebook_loss": codebook_loss,
            "commitment_loss": commitment_loss
        }

In [ ]:
class ConditionalVQVAE(nn.Module):
    def __init__(self,
                 window_size: int,
                 num_features: int,
                 target_indices: Sequence[int],
                 hidden_dims: Sequence[int] = (256, 128, 64),
                 embedding_dim: int = 8,
                 num_embeddings: int = 16,
                 num_latent_tokens: int = 4,
                 commitment_beta: float = 0.25):
        super().__init__()

        self.window_size = window_size
        self.num_features = num_features
        self.target_indices = list(target_indices)
        self.cond_indices = [
            i for i in range(num_features) if i not in self.target_indices
        ]

        self.target_dim = len(self.target_indices)
        self.cond_dim = len(self.cond_indices)
        self.target_input_dim = window_size * self.target_dim
        self.cond_input_dim = window_size * self.cond_dim
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.num_latent_tokens = num_latent_tokens
        self.hidden_dims = tuple(hidden_dims)

        encoder_input_dim = self.target_input_dim + self.cond_input_dim
        encoder_output_dim = num_latent_tokens * embedding_dim
        decoder_input_dim = num_latent_tokens * embedding_dim + \
            self.cond_input_dim
        decoder_output_dim = self.target_input_dim

        self.encoder = self._make_encoder(
            encoder_input_dim, self.hidden_dims, encoder_output_dim)

        self.quantizer = VectorQuantizer(
            num_embeddings, embedding_dim, commitment_beta=commitment_beta)
        self.decoder = self._make_decoder(
            decoder_input_dim, self.hidden_dims, decoder_output_dim)

    @staticmethod
    def _make_encoder(input_dim: int,
                      hidden_dims: Sequence[int],
                      output_dim: int) -> nn.Sequential:
        layers: List[nn.Module] = []
        in_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            in_dim = h
        layers.append(nn.Linear(in_dim, output_dim))
        return nn.Sequential(*layers)

    @staticmethod
    def _make_decoder(input_dim: int,
                      hidden_dims: Sequence[int],
                      output_dim: int) -> nn.Sequential:
        layers: List[nn.Module] = []
        in_dim = input_dim
        for h in reversed(hidden_dims):
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            in_dim = h
        layers.append(nn.Linear(in_dim, output_dim))
        return nn.Sequential(*layers)

    def split_inputs(self, x: torch.Tensor) -> Tuple[torch.Tensor,
                                                     torch.Tensor]:
        target = x[..., self.target_indices]
        cond = x[..., self.cond_indices]
        return target, cond

    @staticmethod
    def _flatten(x: torch.Tensor) -> torch.Tensor:
        return x.reshape(x.size(0), -1)

    def _unflatten_target(self, x: torch.Tensor) -> torch.Tensor:
        return x.reshape(x.size(0), self.window_size, self.target_dim)

    def _reshape_to_tokens(self, z_flat: torch.Tensor) -> torch.Tensor:
        return z_flat.reshape(z_flat.size(0),
                              self.num_latent_tokens,
                              self.embedding_dim)

    def encode_continuous(self,
                          target: torch.Tensor,
                          cond: torch.Tensor) -> torch.Tensor:
        enc_input = torch.cat([self._flatten(target), self._flatten(cond)],
                              dim=1)
        z_flat = self.encoder(enc_input)
        return self._reshape_to_tokens(z_flat)

    def decode(self, z_q: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        z_flat = z_q.reshape(z_q.size(0), -1)
        dec_input = torch.cat([z_flat, self._flatten(cond)], dim=1)
        recon_flat = self.decoder(dec_input)
        return self._unflatten_target(recon_flat)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        target, cond = self.split_inputs(x)
        z_e = self.encode_continuous(target, cond)
        q_out = self.quantizer(z_e)
        recon = self.decode(q_out["z_q_st"], cond)
        recon_loss = F.mse_loss(recon, target)
        total_loss = recon_loss + q_out["vq_loss"]

        return {
            "recon": recon,
            "target": target,
            "cond": cond,
            "total_loss": total_loss,
            "recon_loss": recon_loss,
            "vq_loss": q_out["vq_loss"],
            "codebook_loss": q_out["codebook_loss"],
            "commitment_loss": q_out["commitment_loss"],
            "perplexity": q_out["perplexity"],
            "indices": q_out["indices"],
            "z_e": z_e,
            "z_q": q_out["z_q_st"]
        }

    def reconstruct(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward(x)["recon"]

    def encode_to_indices(self, x: torch.Tensor) -> torch.Tensor:
        target, cond = self.split_inputs(x)
        z_e = self.encode_continuous(target, cond)
        return self.quantizer(z_e)["indices"]

    def decode_code_indices(self,
                            code_indices: torch.Tensor,
                            cond: torch.Tensor) -> torch.Tensor:
        code_indices = code_indices.long()
        z_q = self.quantizer.embedding(code_indices)
        return self.decode(z_q, cond)

    def sample_from_codebook(self,
                             cond: torch.Tensor) -> Tuple[torch.Tensor,
                                                          torch.Tensor]:
        dev = cond.device
        num_samples = cond.size(0)
        indices = torch.randint(
            low=0,
            high=self.num_embeddings,
            size=(num_samples, self.num_latent_tokens),
            device=dev
        )
        samples = self.decode_code_indices(indices, cond)
        return samples, indices

## Conditional VQ-VAE: обучение модели

Реализуем функции обучения и визуализации истории обучения для VQ-VAE.

In [ ]:
def run_cvqvae_epoch(model: nn.Module,
                     loader: DataLoader,
                     device: torch.device,
                     optimizer: Optional[torch.optim.Optimizer] = None
                     ) -> Dict[str, float]:
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_sum = 0.0
    recon_sum = 0.0
    codebook_sum = 0.0
    commitment_sum = 0.0
    perplexity_sum = 0.0
    num_samples = 0

    for (x_batch,) in loader:
        x_batch = x_batch.to(device)

        with torch.set_grad_enabled(is_train):
            out = model(x_batch)
            total_loss = out["total_loss"]
            recon_loss = out["recon_loss"]
            codebook_loss = out["codebook_loss"]
            commitment_loss = out["commitment_loss"]
            perplexity = out["perplexity"]

            if is_train:
                optimizer.zero_grad()
                total_loss.backward()
                optimizer.step()

        batch_size = x_batch.size(0)
        total_sum += total_loss.item() * batch_size
        recon_sum += recon_loss.item() * batch_size
        codebook_sum += codebook_loss.item() * batch_size
        commitment_sum += commitment_loss.item() * batch_size
        perplexity_sum += perplexity.item() * batch_size
        num_samples += batch_size

    return {
        "total": total_sum / num_samples,
        "recon": recon_sum / num_samples,
        "codebook": codebook_sum / num_samples,
        "commitment": commitment_sum / num_samples,
        "perplexity": perplexity_sum / num_samples
    }

In [ ]:
def train_conditional_vqvae(model: nn.Module,
                            train_loader: DataLoader,
                            val_loader: DataLoader,
                            device: torch.device,
                            epochs: int,
                            lr: float) -> Dict[str, List[float]]:
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {
        "train_total": [],
        "train_recon": [],
        "train_codebook": [],
        "train_commitment": [],
        "train_perplexity": [],
        "val_total": [],
        "val_recon": [],
        "val_codebook": [],
        "val_commitment": [],
        "val_perplexity": []
    }

    model.to(device)

    for epoch in range(1, epochs + 1):
        train_metrics = run_cvqvae_epoch(model,
                                         train_loader,
                                         device,
                                         optimizer=optimizer)

        val_metrics = run_cvqvae_epoch(model,
                                       val_loader,
                                       device,
                                       optimizer=None)

        history["train_total"].append(train_metrics["total"])
        history["train_recon"].append(train_metrics["recon"])
        history["train_codebook"].append(train_metrics["codebook"])
        history["train_commitment"].append(train_metrics["commitment"])
        history["train_perplexity"].append(train_metrics["perplexity"])

        history["val_total"].append(val_metrics["total"])
        history["val_recon"].append(val_metrics["recon"])
        history["val_codebook"].append(val_metrics["codebook"])
        history["val_commitment"].append(val_metrics["commitment"])
        history["val_perplexity"].append(val_metrics["perplexity"])

        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:03d}/{epochs} | "
                f"train_total={train_metrics['total']:.4f} | "
                f"val_total={val_metrics['total']:.4f} | "
                f"train_recon={train_metrics['recon']:.4f} | "
                f"train_codebook={train_metrics['codebook']:.4f} | "
                f"train_perplexity={train_metrics['perplexity']:.2f}"
            )

    return history

In [ ]:
def plot_conditional_vq_history(history: Dict[str, List[float]]) -> None:
    fig, axes = plt.subplots(1, 5, figsize=(24, 5))

    axes[0].plot(history["train_total"], label="train")
    axes[0].plot(history["val_total"], label="val")
    axes[0].set_title("Total loss")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(history["train_recon"], label="train")
    axes[1].plot(history["val_recon"], label="val")
    axes[1].set_title("Reconstruction loss")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    axes[2].plot(history["train_codebook"], label="train")
    axes[2].plot(history["val_codebook"], label="val")
    axes[2].set_title("Codebook loss")
    axes[2].grid(alpha=0.3)
    axes[2].legend()

    axes[3].plot(history["train_commitment"], label="train")
    axes[3].plot(history["val_commitment"], label="val")
    axes[3].set_title("Commitment loss")
    axes[3].grid(alpha=0.3)
    axes[3].legend()

    axes[4].plot(history["train_perplexity"], label="train")
    axes[4].plot(history["val_perplexity"], label="val")
    axes[4].set_title("Perplexity")
    axes[4].grid(alpha=0.3)
    axes[4].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
cvqvae = ConditionalVQVAE(
    window_size=WINDOW_SIZE,
    num_features=len(FEATURES),
    target_indices=[TEMP_IDX],
    hidden_dims=VQVAE_CFG["hidden_dims"],
    embedding_dim=VQVAE_CFG["embedding_dim"],
    num_embeddings=VQVAE_CFG["num_embeddings"],
    num_latent_tokens=VQVAE_CFG["num_latent_tokens"],
    commitment_beta=VQVAE_CFG["commitment_beta"]
).to(device)

cvqvae_history = train_conditional_vqvae(
    model=cvqvae,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=VQVAE_CFG["epochs"],
    lr=VQVAE_CFG["lr"]
)

plot_conditional_vq_history(cvqvae_history)

## Conditional VQ-VAE: оценка и анализ кодовой книги

Оценим модель на тестовых окнах, сравним с Continuous VAE по метрикам реконструкции.

In [ ]:
def collect_conditional_vq_outputs(model: nn.Module,
                                   loader: DataLoader,
                                   device: torch.device
                                   ) -> Dict[str, np.ndarray]:
    model.eval()
    recon_list, target_list, indices_list = [], [], []

    with torch.inference_mode():
        for (x_batch,) in loader:
            x_batch = x_batch.to(device, non_blocking=True)
            out = model(x_batch)
            recon_list.append(out["recon"].cpu().numpy())
            target_list.append(out["target"].cpu().numpy())
            indices_list.append(out["indices"].cpu().numpy())

    return {
        "recon": np.concatenate(recon_list, axis=0),
        "target": np.concatenate(target_list, axis=0),
        "indices": np.concatenate(indices_list, axis=0)
    }

In [ ]:
def evaluate_conditional_vqvae(model: nn.Module,
                               loader: DataLoader,
                               device: torch.device,
                               scaler: StandardScaler,
                               target_pos_in_scaler: int
                               ) -> Dict[str, object]:
    outputs = collect_conditional_vq_outputs(model, loader, device)

    true_windows = inverse_scale_feature(
        outputs["target"],
        scaler,
        target_pos_in_scaler
    )
    pred_windows = inverse_scale_feature(
        outputs["recon"],
        scaler,
        target_pos_in_scaler
    )

    if true_windows.ndim == 3 and true_windows.shape[-1] == 1:
        true_windows = true_windows[..., 0]

    if pred_windows.ndim == 3 and pred_windows.shape[-1] == 1:
        pred_windows = pred_windows[..., 0]

    results = {
        "true": true_windows,
        "pred": pred_windows
    }

    y_true = true_windows.reshape(-1)
    y_pred = pred_windows.reshape(-1)
    metrics = regression_metrics(y_true, y_pred)

    return {
        "outputs": outputs,
        "results": results,
        "metrics": metrics
    }

In [ ]:
cvq_eval = evaluate_conditional_vqvae(
    model=cvqvae,
    loader=test_loader,
    device=device,
    scaler=scaler,
    target_pos_in_scaler=TEMP_POS_IN_SCALER
)

comparison_df = pd.DataFrame([
    {"Model": "Conditional VAE", **test_results["metrics"]},
    {"Model": "Conditional VQ-VAE", **cvq_eval["metrics"]}
])

display(comparison_df)

In [ ]:
plot_window_reconstructions(
    cvq_eval["results"],
    "Conditional VQ-VAE",
    starts=test_starts,
    max_examples=5
)

In [ ]:
plot_scatter_and_errors(
    cvq_eval["results"]["true"].reshape(-1),
    cvq_eval["results"]["pred"].reshape(-1),
    "Conditional VQ-VAE"
)

Теперь проанализируем частоту использования кодов и покажем типичные окна для наиболее популярных кодов.

In [ ]:
def collect_vq_code_indices(model: nn.Module,
                            loader: DataLoader,
                            device: torch.device) -> np.ndarray:
    model.eval()
    indices_list = []

    with torch.inference_mode():
        for (x_batch,) in loader:
            x_batch = x_batch.to(device, non_blocking=True)
            indices = model.encode_to_indices(x_batch)
            indices_list.append(indices.cpu().numpy())

    return np.concatenate(indices_list, axis=0)

In [ ]:
def get_code_usage_df(code_indices: np.ndarray,
                      num_embeddings: int) -> pd.DataFrame:
    flat_indices = np.asarray(code_indices).reshape(-1)
    counts = np.bincount(flat_indices, minlength=num_embeddings)
    shares = counts / counts.sum()
    return pd.DataFrame({
        "code": np.arange(num_embeddings),
        "count": counts,
        "share": shares
    })

In [ ]:
def plot_code_usage_comparison(train_usage_df: pd.DataFrame,
                               val_usage_df: pd.DataFrame,
                               test_usage_df: pd.DataFrame) -> None:
    x = train_usage_df["code"].values
    width = 0.28

    plt.figure(figsize=(15, 5))
    plt.bar(x - width, train_usage_df["share"], width=width, label="train")
    plt.bar(x, val_usage_df["share"], width=width, label="val")
    plt.bar(x + width, test_usage_df["share"], width=width, label="test")
    plt.title("Сравнение использования кодов")
    plt.xlabel("Индекс кода")
    plt.ylabel("Доля использований")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.show()

In [ ]:
train_code_indices = collect_vq_code_indices(cvqvae, train_loader, device)
val_code_indices = collect_vq_code_indices(cvqvae, val_loader, device)
test_code_indices = collect_vq_code_indices(cvqvae, test_loader, device)

train_usage_df = get_code_usage_df(train_code_indices, cvqvae.num_embeddings)
val_usage_df = get_code_usage_df(val_code_indices, cvqvae.num_embeddings)
test_usage_df = get_code_usage_df(test_code_indices, cvqvae.num_embeddings)

plot_code_usage_comparison(train_usage_df, val_usage_df, test_usage_df)

In [ ]:
def plot_windows_for_code(original_windows: np.ndarray,
                          code_indices: np.ndarray,
                          code: int,
                          max_windows: int = 50) -> None:
    original_windows = np.asarray(original_windows)
    code_indices = np.asarray(code_indices)
    flat_codes = code_indices.reshape(code_indices.shape[0], -1)

    mask = (flat_codes == code).any(axis=1)
    matched = original_windows[mask]

    if len(matched) == 0:
        print(f"Для кода {code} окон не найдено.")
        return

    matched_to_plot = matched[:max_windows]
    median_curve = np.median(matched_to_plot, axis=0)
    q25 = np.percentile(matched_to_plot, 25, axis=0)
    q75 = np.percentile(matched_to_plot, 75, axis=0)
    x = np.arange(matched_to_plot.shape[1])

    plt.figure(figsize=(12, 5))
    for window in matched_to_plot:
        plt.plot(x, window, color="steelblue", alpha=0.12, linewidth=1)

    plt.fill_between(x, q25, q75, color="lightskyblue",
                     alpha=0.35, label="25-75 перцентили")
    plt.plot(x, median_curve, color="darkblue", linewidth=2.5, label="Медиана")
    plt.title(f"Окна для кода {code} | найдено: {len(matched)} " +
              f"| показано: {len(matched_to_plot)}")
    plt.xlabel("Шаг внутри окна")
    plt.ylabel(TARGET)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

In [ ]:
print("Всего кодов в словаре:", cvqvae.num_embeddings)
print("Использовано кодов на train:", int((train_usage_df["count"] > 0).sum()))
print("Использовано кодов на val:  ", int((val_usage_df["count"] > 0).sum()))
print("Использовано кодов на test: ", int((test_usage_df["count"] > 0).sum()))

top_codes = test_usage_df.sort_values(
    "count", ascending=False).head(3)["code"].tolist()
print("Три самых частых кода на test:", top_codes)


In [ ]:
for code in top_codes:
    plot_windows_for_code(cvq_eval["results"]["true"],
                          test_code_indices,
                          code=code,
                          max_windows=50)

## Склейка окон и разрывы на стыках

Независимая реконструкция окон может давать скачки на границах при сборке длинного ряда. Для смягчения этого эффекта применим overlap-add усреднение и визуализируем сравнение непрерывного ряда с реконструкциями двух моделей.


In [ ]:
def stitch_windows_mean(windows: np.ndarray,
                        starts: np.ndarray,
                        total_length: int) -> np.ndarray:
    result = np.zeros(total_length, dtype=np.float32)
    counts = np.zeros(total_length, dtype=np.float32)
    window_size = windows.shape[1]

    for window, start in zip(windows, starts):
        end = min(start + window_size, total_length)
        valid_len = end - start
        result[start:end] += window[:valid_len]
        counts[start:end] += 1

    mask = counts > 0
    result[mask] /= counts[mask]
    return result

In [ ]:
def plot_stitched_model_comparison(cvae_model: nn.Module,
                                   cvqvae_model: nn.Module,
                                   test_scaled: np.ndarray,
                                   test_original_temp: np.ndarray,
                                   dates: np.ndarray,
                                   window_size: int,
                                   scaler: StandardScaler,
                                   target_pos_in_scaler: int,
                                   device: torch.device,
                                   fragment_len: int = 220,
                                   batch_size: int = 256) -> None:
    windows_data = create_sliding_windows(test_scaled,
                                          window_size=window_size,
                                          stride=1)
    windows = windows_data["windows"]
    starts = windows_data["starts"]
    ends = windows_data["ends"]

    loader = DataLoader(
        TensorDataset(torch.from_numpy(np.asarray(windows, dtype=np.float32))),
        batch_size=batch_size,
        shuffle=False
    )

    cvae_model.eval()
    cvqvae_model.eval()
    cvae_recon_list, cvqvae_recon_list = [], []

    with torch.inference_mode():
        for (x_batch,) in loader:
            x_batch = x_batch.to(device, non_blocking=True)
            cvae_recon_batch, _, _, _ = cvae_model(x_batch, sample=False)
            cvae_recon_list.append(cvae_recon_batch.cpu().numpy())

            cvq_out = cvqvae_model(x_batch)
            cvqvae_recon_list.append(cvq_out["recon"].cpu().numpy())

    cvae_recon_scaled = np.concatenate(cvae_recon_list, axis=0)
    cvqvae_recon_scaled = np.concatenate(cvqvae_recon_list, axis=0)

    cvae_recon = inverse_scale_feature(
        cvae_recon_scaled, scaler, target_pos_in_scaler)
    cvqvae_recon = inverse_scale_feature(
        cvqvae_recon_scaled, scaler, target_pos_in_scaler)

    if cvae_recon.ndim == 3 and cvae_recon.shape[-1] == 1:
        cvae_recon = cvae_recon[..., 0]
    if cvqvae_recon.ndim == 3 and cvqvae_recon.shape[-1] == 1:
        cvqvae_recon = cvqvae_recon[..., 0]

    stitched_cvae = stitch_windows_mean(
        cvae_recon, starts, total_length=len(test_original_temp))
    stitched_cvqvae = stitch_windows_mean(
        cvqvae_recon, starts, total_length=len(test_original_temp))

    covered_len = ends[-1] if len(ends) > 0 else 0
    final_len = min(fragment_len, covered_len, len(test_original_temp))

    plt.figure(figsize=(16, 6))
    plt.plot(dates[:final_len],
             test_original_temp[:final_len],
             label="Исходный ряд",
             color="black",
             linewidth=2)

    plt.plot(dates[:final_len],
             stitched_cvae[:final_len],
             label="Conditional VAE",
             color="crimson",
             alpha=0.9)

    plt.plot(dates[:final_len],
             stitched_cvqvae[:final_len],
             label="Conditional VQ-VAE",
             color="royalblue",
             alpha=0.9)

    plt.title("Склейка реконструированных окон в полный ряд")
    plt.xlabel("Дата")
    plt.ylabel(TARGET)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
test_original_temp = inverse_scale_feature(
    X_test[:, [TEMP_IDX]], scaler, TEMP_POS_IN_SCALER).reshape(-1)

plot_stitched_model_comparison(
    cvae_model=cvae,
    cvqvae_model=cvqvae,
    test_scaled=X_test,
    test_original_temp=test_original_temp,
    dates=test_dates,
    window_size=WINDOW_SIZE,
    scaler=scaler,
    target_pos_in_scaler=TEMP_POS_IN_SCALER,
    device=device,
    fragment_len=PLOT_CFG["fragment_len"]
)

## Индивидуальное задание

Опираясь на представленное исследование и используя его в качестве образца, **выполните аналогичные действия с выбранным Вами датасетом**.

В процессе работы:

1. Выберите датасет, содержащий **временной ряд**. Датасеты можно найти, например, на [Kaggle](https://www.kaggle.com/datasets) и [UC Irvine Machine Learning Repository](https://archive.ics.uci.edu/ml/index.php). Датасет должен содержать целевую переменную и **хотя бы один** дополнительный признак.

2. Выполните **разведочный анализ** датасета: исследуйте временную структуру, сезонность, тренды. Выполните требуемые **преобразования признаков** (масштабирование, бинаризация и др.).

3. Разделите датасет на **обучающую**, **валидационную** и **тестовую** части с сохранением временного порядка.

4. Перейдите от исходного временного ряда к набору **скользящих окон фиксированной длины** и подготовьте данные для обучения генеративных моделей.

5. Реализуйте и обучите **условный вариационный автокодировщик** (`Conditional VAE`) для реконструкции окна целевой переменной по латентному представлению и остальным признакам в окне.

6. Постройте графики **истории обучения Conditional VAE**, включая общую функцию потерь, ошибку реконструкции, KL-слагаемое и изменение коэффициента $\beta$.

7. Выполните **оценку Conditional VAE**: используйте валидационную выборку для контроля обучения и подбора гиперпараметров, а тестовую -- для итоговой оценки качества реконструкции, визуализации результатов и вычисления метрик.

8. Исследуйте влияние коэффициента $\beta$ в модели **$\beta$-VAE**, обучите несколько вариантов модели при разных значениях $\beta$ и сравните их по ошибке реконструкции, KL-слагаемому и итоговым метрикам.

9. Продемонстрируйте свойства непрерывного латентного пространства Conditional VAE, выполнив **интерполяцию** между двумя окнами и **генерацию** окон из априорного распределения.

10. Реализуйте и обучите **условный VQ-VAE** (Conditional VQ-VAE) с дискретной кодовой книгой для реконструкции целевой переменной по временному окну.

11. Постройте графики **истории обучения Conditional VQ-VAE**, включая общую функцию потерь, ошибку реконструкции, потери кодовой книги, commitment loss и перплексию.

12. **Оцените Conditional VQ-VAE** на валидационной и тестовой выборках: по валидационной выборке контролируйте обучение и подбирайте гиперпараметры, по тестовой -- выполните итоговую оценку качества реконструкции и визуализацию результатов.

13. Выполните **анализ кодовой книги** Conditional VQ-VAE, исследуйте частоты использования кодов на обучающей, валидационной и тестовой выборках, определите число реально используемых кодов и проанализируйте значение перплексии.

14. **Сравните** Conditional VAE и Conditional VQ-VAE по качеству реконструкции, визуальному виду восстановленных окон, значениям метрик и свойствам латентного пространства.

15. **Исследуйте** восстановление длинного фрагмента временного ряда по реконструированным окнам и **примените метод overlap-add** для сглаживания границ между соседними окнами.

16. Сделайте **выводы** по работе.



## Вопросы для подготовки к отчёту

1. Понятие дискриминативных и генеративных моделей. Классификация и основные свойства глубоких генеративных моделей.  
2. Понятие автокодировщиков. Классический и вариационный автокодировщик. Идея модели VAE. Латентное пространство AE и VAE.
3. Вариационный автокодировщик. Семейство вариационных распределений в VAE. Нижняя вариационная граница правдоподобия (ELBO) для VAE и интерпретация её слагаемых. Коллапс апостериорного распределения.  
4. Запись ELBO для частного случая нормальных распределений. Функция потерь VAE при нормальном априорном распределении.  
5. Обучение VAE. Задачи реконструкции и генерации объектов. Особенности объектов, сгенерированных VAE. Оценка результатов генерации, perceptual loss. Условный VAE и его функция потерь.
6. Распутанность латентных представлений в VAE. Модель $\beta$-VAE и её функция потерь. Распутывание признаков в $\beta$-VAE.
7. Недостатки классических VAE. VAE с дискретным латентным пространством. Два подхода к построению VAE: Gumbel-Softmax и VQ-VAE.
8. VQ-VAE. Векторная квантизация. Кодовые векторы и кодовая книга. Функция потерь модели VQ-VAE и её основные слагаемые. Обучение VQ-VAE, функция ошибки и роль каждого слагаемого. Straight-Through Estimator. Обновление словаря с помощью EMA (Exponential Moving Average). Коллапс кодовой книги.
9. Проблемы семплирования в VQ-VAE. Обучение априорного распределения в дискретном латентном пространстве. Применение авторегрессионной модели.  
10. Практические аспекты обучения VAE и VQ-VAE. Выбор размерности латентного пространства, коэффициента $\beta$, размера кодовой книги, числа латентных токенов. Перплексия и использование кодовой книги как диагностические характеристики VQ-VAE.
11. Иерархические дискретные латентные представления. Модель VQ-VAE-2, её архитектура, многоуровневое кодирование и преимущества по сравнению с VQ-VAE.
12. Современные применения VAE и VQ-VAE. Роль непрерывных и дискретных латентных представлений в задачах реконструкции, генерации и сжатия изображений, речи и временных рядов.


## Список рекомендуемой литературы

1. **Kingma, D. P.** Auto-Encoding Variational Bayes / D. P. Kingma, M. Welling // arXiv preprint arXiv:1312.6114. – 2013. – URL: https://arxiv.org/abs/1312.6114 (дата обращения: 16.03.2026). – Текст : электронный.
2. **Rezende, D. J.** Stochastic Backpropagation and Approximate Inference in Deep Generative Models / D. J. Rezende, S. Mohamed, D. Wierstra // Proceedings of the 31st International Conference on Machine Learning (ICML). – 2014. – Vol. 32. – P. 1278–1286. – Текст : непосредственный.
3. **Van den Oord, A.** Neural Discrete Representation Learning / A. Van den Oord, O. Vinyals, K. Kavukcuoglu // Advances in Neural Information Processing Systems (NeurIPS). – 2017. – Vol. 30. – P. 6306–6315. – Текст : непосредственный.
4. **Razavi, A.** Generating Diverse High-Fidelity Images with VQ-VAE-2 / A. Razavi, A. Van den Oord, O. Vinyals // Advances in Neural Information Processing Systems (NeurIPS). – 2019. – Vol. 32. – P. 14866–14876. – Текст : непосредственный.
5. **Goodfellow, I.** Deep Learning / I. Goodfellow, Y. Bengio, A. Courville. – Cambridge : MIT Press, 2016. – 800 p. – Текст : непосредственный.
6. **Bishop, C. M.** Pattern Recognition and Machine Learning / C. M. Bishop. – New York : Springer, 2006. – 738 p. – Текст : непосредственный.
7. **Doersch, C.** Tutorial on Variational Autoencoders / C. Doersch // arXiv preprint arXiv:1606.05908. – 2016. – URL: https://arxiv.org/abs/1606.05908 (дата обращения: 16.03.2026). – Текст : электронный.
8. **PyTorch documentation** [Электронный ресурс] : официальный сайт. – URL: https://pytorch.org/docs/stable/ (дата обращения: 16.03.2026).
